# llama32_3b — Azerbaijani STS

Zero-shot evaluation and QLoRA adaptation. Run cells in order on a CUDA GPU with Internet enabled. Data download from this GitHub repository; no Kaggle dataset attachment is needed.

Set `HF_TOKEN` in the environment, a local ignored `.env`, Kaggle Secrets, or Colab Secrets. Accept the checkpoint access terms on Hugging Face when required. Never paste credentials into a cell.

`AZSEMBENCH_REPO`, `AZSEMBENCH_REF`, `AZSEMBENCH_OUTPUT_DIR` and optional `AZSEMBENCH_DATA_SHA256` configure the dataset and output location. Use a fresh output directory for a new run. Saved historical metrics are under `results/`; outputs in this notebook are cleared.


In [ ]:
%pip -q install -U "transformers==4.57.1" "trl==0.24.0" "peft==0.17.1" "accelerate>=1.10.0" "bitsandbytes>=0.45.0" "datasets>=3.0.0,<4.0.0" "scikit-learn>=1.4" "scipy>=1.11,<1.17" "huggingface_hub>=0.27" python-dotenv


## Repository data and private configuration

In [ ]:
import os
import hashlib
from pathlib import Path
from urllib.request import urlopen
from dotenv import load_dotenv

load_dotenv()
REPO = os.getenv("AZSEMBENCH_REPO", "MehribanAliyeva/azsembench")
REF = os.getenv("AZSEMBENCH_REF", "main")
OUTPUT_ROOT = Path(os.getenv("AZSEMBENCH_OUTPUT_DIR", "outputs")).expanduser()
DATA_DIR = OUTPUT_ROOT / "datasets"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_URL = f"https://raw.githubusercontent.com/{REPO}/{REF}/data/model_ready/core_model_ready.csv"
CSV_PATH = DATA_DIR / "core_model_ready.csv"
with urlopen(DATA_URL, timeout=120) as response:
    payload = response.read()
CSV_PATH.write_bytes(payload)
DATA_SHA256 = hashlib.sha256(payload).hexdigest()
expected_hash = os.getenv("AZSEMBENCH_DATA_SHA256")
if expected_hash and DATA_SHA256 != expected_hash:
    raise ValueError("Dataset hash differs from AZSEMBENCH_DATA_SHA256.")
print("Dataset:", DATA_URL)
print("SHA-256:", DATA_SHA256)

token = os.getenv("HF_TOKEN", "").strip()
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except Exception:
        pass
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
if token:
    os.environ["HF_TOKEN"] = token


In [ ]:
import re,json,math,shutil,warnings
from pathlib import Path
import numpy as np,pandas as pd,torch
from datasets import Dataset
from scipy.stats import pearsonr,spearmanr,kendalltau
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig,set_seed
from peft import LoraConfig,prepare_model_for_kbit_training
from trl import SFTTrainer,SFTConfig

warnings.filterwarnings("ignore")

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU.")

GPU_NAME=torch.cuda.get_device_name(0)
CC_MAJOR,CC_MINOR=torch.cuda.get_device_capability()
USE_BF16=CC_MAJOR>=8
COMPUTE_DTYPE=torch.bfloat16 if USE_BF16 else torch.float16

SEED=42
set_seed(SEED)

MODEL_ID="meta-llama/Llama-3.2-3B-Instruct"
RUN_DIR = OUTPUT_ROOT / "llm_sts_llama"
RUN_DIR.mkdir(parents=True,exist_ok=True)

print("GPU:",GPU_NAME,"CC:",f"{CC_MAJOR}.{CC_MINOR}","BF16:",USE_BF16)


In [ ]:

MAX_SEQ_LEN=384
MAX_NEW_TOKENS=8

NUM_EPOCHS=2
LEARNING_RATE=2e-4
TRAIN_BATCH_SIZE=1
EVAL_BATCH_SIZE=1
GRAD_ACCUM=8

LORA_R=16
LORA_ALPHA=32
LORA_DROPOUT=0.05

SMOKE_TEST=False
SAVE_EVERY=50

# A new output directory is required when the data or model configuration changes.
RUN_METADATA = {
    "model_id": MODEL_ID, "seed": SEED, "data_url": DATA_URL,
    "data_sha256": DATA_SHA256, "max_seq_len": MAX_SEQ_LEN,
    "epochs": NUM_EPOCHS, "learning_rate": LEARNING_RATE,
    "grad_accum": GRAD_ACCUM, "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
    "train_subset_n": globals().get("TRAIN_SUBSET_N"),
}
metadata_path = RUN_DIR / "run_metadata.json"
if metadata_path.exists():
    if json.loads(metadata_path.read_text()) != RUN_METADATA:
        raise RuntimeError("Output configuration changed; choose a fresh AZSEMBENCH_OUTPUT_DIR.")
metadata_path.write_text(json.dumps(RUN_METADATA, indent=2) + "\n")


## Hugging Face login

In [ ]:
# Access is checked without storing a credential in notebook source.
from huggingface_hub import hf_hub_download
if not token:
    raise RuntimeError("Set HF_TOKEN using an environment variable or notebook secret.")
hf_hub_download(repo_id=MODEL_ID, filename="config.json", token=token)
print("Model access confirmed.")


## Load STS data

In [ ]:
raw = pd.read_csv(CSV_PATH, dtype={"source_id": str})
required = {"source_id", "task", "split", "sentence1_az", "sentence2_az", "label"}
if required - set(raw.columns):
    raise ValueError(f"Missing columns: {sorted(required - set(raw.columns))}")
raw = raw.loc[raw["task"].eq("semantic_similarity")].copy()
if raw[["sentence1_az", "sentence2_az", "label", "source_id"]].isna().any().any():
    raise ValueError("STS examples must have IDs, two translations and a label.")
labels = pd.to_numeric(raw["label"], errors="raise")
if not labels.between(0, 5).all():
    raise ValueError("STS labels must be between 0 and 5.")
df = pd.DataFrame({"id": raw["source_id"].astype(str),
    "sentence1": raw["sentence1_az"].astype(str),
    "sentence2": raw["sentence2_az"].astype(str),
    "label": labels.astype(float), "split": raw["split"], "task": "sts"}).reset_index(drop=True)
if df["id"].duplicated().any():
    raise ValueError("Duplicate STS IDs.")
train_df = df.loc[df["split"].eq("train")].reset_index(drop=True)
dev_df = df.loc[df["split"].eq("dev")].reset_index(drop=True)
test_df = df.loc[df["split"].eq("test")].reset_index(drop=True)
if any(part.empty for part in [train_df, dev_df, test_df]):
    raise ValueError("Expected non-empty train, dev and test partitions.")
if SMOKE_TEST:
    train_df, dev_df, test_df = train_df.head(64), dev_df.head(32), test_df.head(20)
print({"train": len(train_df), "dev": len(dev_df), "test": len(test_df)})


## Load Llama 3.2 3B in 4-bit

In [ ]:
bnb=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer=AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=token,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token=tokenizer.eos_token

tokenizer.padding_side="left"

model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=token,
    device_map="auto",
    quantization_config=bnb,
    dtype=COMPUTE_DTYPE,
)

model.eval()
print("Loaded:",MODEL_ID)


## Prompt

In [ ]:
SYSTEM_PROMPT=(
    "Sən Azərbaycan dilində cümlələr arasında semantik oxşarlığı qiymətləndirirsən. "
    "Cavabda yalnız bir ədəd qaytar."
)

def user_prompt(s1,s2):
    return (
        "Aşağıdakı iki cümlənin semantik oxşarlığını 0.0-dan 5.0-a qədər qiymətləndir.\n"
        "0.0 = tamamilə fərqli məna\n"
        "5.0 = eyni məna\n"
        "Yalnız bir ədəd qaytar. Heç bir izah yazma.\n\n"
        f"Cümlə 1: {s1}\n"
        f"Cümlə 2: {s2}\n"
        "Qiymət:"
    )

def make_messages(s1,s2,answer=None):
    msgs=[
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":user_prompt(s1,s2)},
    ]
    if answer is not None:
        msgs.append({"role":"assistant","content":answer})
    return msgs

def target_text(y):
    return f"{float(y):.3f}".rstrip("0").rstrip(".")


## Zero-shot scoring

In [ ]:
@torch.inference_mode()
def generate_one(s1,s2):
    inputs=tokenizer.apply_chat_template(
        make_messages(s1,s2),
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    input_len=inputs["input_ids"].shape[-1]

    out=model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(
        out[0,input_len:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()

def parse_sts(text):
    if text is None:
        return np.nan

    text=str(text).strip().replace(",",".")

    nums=re.findall(
        r"(?<!\d)[-+]?(?:\d+(?:\.\d+)?|\.\d+)(?!\d)",
        text,
    )

    vals=[]
    for n in nums:
        try:
            x=float(n)
            if 0<=x<=5:
                vals.append(x)
        except Exception:
            pass

    return vals[-1] if vals else np.nan


## Mandatory 5-example smoke test

In [ ]:
good=0

for i in range(min(5,len(test_df))):
    r=test_df.iloc[i]

    raw_out=generate_one(
        r.sentence1,
        r.sentence2,
    )

    pred=parse_sts(raw_out)

    print("RAW:",repr(raw_out))
    print("PARSED:",pred,"GOLD:",r.label)
    print("-"*70)

    if not pd.isna(pred):
        good+=1

if good<min(5,len(test_df)):
    raise RuntimeError(
        f"Smoke test failed: {good}/5 parsed."
    )

print("Smoke test passed:",good,"/5")


## Resume-safe inference

In [ ]:
def run_resumable(source_df,out_path,condition):
    out_path=Path(out_path)

    if out_path.exists():
        old=pd.read_csv(out_path)
        done_ids=set(old.id.astype(str))
    else:
        done_ids=set()

    pending=source_df[
        ~source_df.id.astype(str).isin(done_ids)
    ].copy()

    buffer=[]

    for j,(_,r) in enumerate(pending.iterrows(),1):
        raw_out=generate_one(
            r.sentence1,
            r.sentence2,
        )

        pred=parse_sts(raw_out)

        buffer.append({
            "id":str(r.id),
            "sentence1":r.sentence1,
            "sentence2":r.sentence2,
            "gold":float(r.label),
            "raw_output":raw_out,
            "prediction":pred,
            "condition":condition,
            "model":"llama32_3b",
            "model_id":MODEL_ID,
            "seed":SEED,
        })

        if j%SAVE_EVERY==0 or j==len(pending):
            new=pd.DataFrame(buffer)

            if out_path.exists():
                old=pd.read_csv(out_path)
                combined=pd.concat(
                    [old,new],
                    ignore_index=True,
                ).drop_duplicates(
                    "id",
                    keep="last",
                )
            else:
                combined=new

            combined.to_csv(
                out_path,
                index=False,
            )

            buffer=[]

            print(
                condition,
                len(combined),
                "/",
                len(source_df),
                "saved",
            )

    return pd.read_csv(out_path)


## Zero-shot STS

In [ ]:
ZERO_PATH=RUN_DIR/"zero_shot_predictions.csv"

zero_preds=run_resumable(
    test_df,
    ZERO_PATH,
    "zero_shot",
)


## Metrics

In [ ]:
def safe_corr(fn,y,p):
    try:
        return float(fn(y,p)[0])
    except Exception:
        return np.nan

def metrics(pred_df):
    d=pred_df.dropna(
        subset=["prediction"]
    ).copy()

    y=d.gold.astype(float).to_numpy()
    p=d.prediction.astype(float).to_numpy()

    return {
        "n":len(pred_df),
        "parsed_n":len(d),
        "parse_coverage":len(d)/max(len(pred_df),1),
        "pearson":safe_corr(pearsonr,y,p) if len(d)>1 else np.nan,
        "spearman":safe_corr(spearmanr,y,p) if len(d)>1 else np.nan,
        "kendall":safe_corr(kendalltau,y,p) if len(d)>1 else np.nan,
        "mae":mean_absolute_error(y,p) if len(d) else np.nan,
        "rmse":math.sqrt(mean_squared_error(y,p)) if len(d) else np.nan,
        "r2":r2_score(y,p) if len(d)>1 else np.nan,
    }

zero_metrics=metrics(zero_preds)

zero_metrics.update({
    "model":"llama32_3b",
    "model_id":MODEL_ID,
    "task":"sts",
    "condition":"zero_shot",
    "seed":SEED,
})

print(zero_metrics)

with open(
    RUN_DIR/"zero_shot_metrics.json",
    "w",
) as f:
    json.dump(
        zero_metrics,
        f,
        indent=2,
        allow_nan=True,
    )


## Build QLoRA SFT dataset

In [ ]:
def make_text(r):
    return tokenizer.apply_chat_template(
        make_messages(
            r.sentence1,
            r.sentence2,
            answer=target_text(r.label),
        ),
        tokenize=False,
        add_generation_prompt=False,
    )

train_ds=Dataset.from_dict({
    "text":[
        make_text(r)
        for _,r in train_df.iterrows()
    ]
})

dev_ds=Dataset.from_dict({
    "text":[
        make_text(r)
        for _,r in dev_df.iterrows()
    ]
})

print(train_ds)
print(dev_ds)


## QLoRA fine-tuning

In [ ]:
model.config.use_cache=False

model=prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora=LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

args=SFTConfig(
    output_dir=str(RUN_DIR/"trainer"),

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,

    lr_scheduler_type="cosine",
    warmup_steps=20,

    logging_steps=10,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    optim="paged_adamw_8bit",

    fp16=not USE_BF16,
    bf16=USE_BF16,

    gradient_checkpointing=True,

    max_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field="text",

    report_to="none",

    seed=SEED,
    data_seed=SEED,
)

trainer=SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    processing_class=tokenizer,
    peft_config=lora,
)

trainer.train()

ADAPTER_DIR=RUN_DIR/"adapter"

trainer.save_model(
    str(ADAPTER_DIR)
)

tokenizer.save_pretrained(
    str(ADAPTER_DIR)
)

pd.DataFrame(
    trainer.state.log_history
).to_csv(
    RUN_DIR/"trainer_log.csv",
    index=False,
)

model=trainer.model
model.config.use_cache=True
model.eval()

print("Fine-tuning complete.")


## Fine-tuned STS evaluation

In [ ]:
FT_PATH=RUN_DIR/"finetuned_predictions.csv"

fine_preds=run_resumable(
    test_df,
    FT_PATH,
    "qlora_finetuned",
)

fine_metrics=metrics(fine_preds)

fine_metrics.update({
    "model":"llama32_3b",
    "model_id":MODEL_ID,
    "task":"sts",
    "condition":"qlora_finetuned",
    "seed":SEED,
})

with open(
    RUN_DIR/"finetuned_metrics.json",
    "w",
) as f:
    json.dump(
        fine_metrics,
        f,
        indent=2,
        allow_nan=True,
    )

comparison=pd.DataFrame([
    zero_metrics,
    fine_metrics,
])

comparison.to_csv(
    RUN_DIR/"all_metrics.csv",
    index=False,
)

display(comparison)


## Zip outputs

In [ ]:
print("Results and adapter saved under:", RUN_DIR.resolve())
